In [1]:
import sqlalchemy as db

In [2]:
engine = db.create_engine('sqlite:///mydbtest.db')

connection = engine.connect()
metadata = db.MetaData()

In [3]:
students = db.Table(
    'Student', metadata,
    db.Column('id', db.Integer, primary_key = True),
    db.Column('name', db.Text)
)

students_in_class = db.Table(
    'Student_in_class', metadata,
    db.Column('id', db.Integer, primary_key = True),
    db.Column('student_id', db.Integer),
    db.Column('class_name', db.Text)
)

metadata.create_all(engine)

In [4]:
insert_student_1 = students.insert().values([
    {'id': 1, 'name': 'John Doe'},
    {'id': 2, 'name': 'John Smith'},
    {'id': 3, 'name': 'Mary Jane'}
])
connection.execute(insert_student_1)

insert_student_in_calss_1 = students_in_class.insert().values([
    {'id': 1, 'student_id': 1, 'class_name': '11 A'},
    {'id': 2, 'student_id': 2, 'class_name': '11 A'},
    {'id': 3, 'student_id': 3, 'class_name': '11 B'}
])
connection.execute(insert_student_in_calss_1)

In [5]:
student_select_all = db.select(students)
student_select_all_result = connection.execute(student_select_all)
student_select_all_result.fetchall()

[(1, 'John Doe'), (2, 'John Smith'), (3, 'Mary Jane')]

In [6]:
student_in_class_select_all = db.select(students_in_class)
student_in_class_select_all_result = connection.execute(student_in_class_select_all)
student_in_class_select_all_result.fetchall()

[(1, 1, '11 A'), (2, 2, '11 A'), (3, 3, '11 B')]

In [7]:
select_11_a_students_query = db.select(students_in_class).where(students_in_class.columns.class_name == '11 A')
select_11_a_students_result = connection.execute(select_11_a_students_query)
select_11_a_students_result.fetchall()

[(1, 1, '11 A'), (2, 2, '11 A')]

In [10]:
group_by_class_query = db.select(
    students_in_class.columns.class_name, 
    db.func.count(students_in_class.columns.class_name)
).group_by(
    students_in_class.columns.class_name
)
group_by_class_result = connection.execute(group_by_class_query)
group_by_class_result.fetchall()

[('11 A', 2), ('11 B', 1)]

In [11]:
order_by_student_query = db.select(students.columns.name).order_by(students.columns.name)
order_by_student_result = connection.execute(order_by_student_query)
order_by_student_result.fetchall()

[('John Doe',), ('John Smith',), ('Mary Jane',)]

In [14]:
insert_student_2 = students.insert().values([
    {'id': 4, 'name': 'Jane Wood'}
])
connection.execute(insert_student_2)

insert_student_in_calss_2 = students_in_class.insert().values([
    {'id': 4, 'student_id': 5, 'class_name': '11 C'}
])
connection.execute(insert_student_in_calss_2)

In [17]:
full_outer_join_query = db.select(
    students.columns.id, 
    students.columns.name, 
    students_in_class.columns.class_name
).select_from(
    students
).join(students_in_class, students_in_class.columns.student_id == students.columns.id, full=True)

full_outer_join_result = connection.execute(full_outer_join_query)
full_outer_join_result.fetchall()

[(1, 'John Doe', '11 A'),
 (2, 'John Smith', '11 A'),
 (3, 'Mary Jane', '11 B'),
 (4, 'Jane Wood', None),
 (None, None, '11 C')]

In [19]:
inner_join_query = db.select(
    students.columns.id, 
    students.columns.name, 
    students_in_class.columns.class_name
).select_from(
    students
).join(students_in_class, students_in_class.columns.student_id == students.columns.id)

inner_join_result = connection.execute(inner_join_query)
inner_join_result.fetchall()

[(1, 'John Doe', '11 A'), (2, 'John Smith', '11 A'), (3, 'Mary Jane', '11 B')]

In [20]:
left_join_query = db.select(
    students.columns.id, 
    students.columns.name, 
    students_in_class.columns.class_name
).select_from(
    students
).join(students_in_class, students_in_class.columns.student_id == students.columns.id, isouter=True)

left_join_result = connection.execute(left_join_query)
left_join_result.fetchall()

[(1, 'John Doe', '11 A'),
 (2, 'John Smith', '11 A'),
 (3, 'Mary Jane', '11 B'),
 (4, 'Jane Wood', None)]

In [23]:
right_join_query = db.select(
    students_in_class.columns.id, 
    students.columns.name, 
    students_in_class.columns.class_name
).select_from(
    students_in_class
).join(students, students_in_class.columns.student_id == students.columns.id, isouter=True)

right_join_result = connection.execute(right_join_query)
right_join_result.fetchall()

[(1, 'John Doe', '11 A'),
 (2, 'John Smith', '11 A'),
 (3, 'Mary Jane', '11 B'),
 (4, None, '11 C')]

In [26]:
select_11_a_students_query = db.select(
    students_in_class.columns.id
).where(
    students_in_class.columns.class_name == '11 A'
)

subquery_final_query = db.select(students.columns.name).where(students.columns.id.in_(select_11_a_students_query))

subquery_final_query_result = connection.execute(subquery_final_query)
subquery_final_query_result.fetchall()

[('John Doe',), ('John Smith',)]